# Car noise removal: full ensense dataset

This notebook just runs the `carnoise` package. All the processing code lives in `src/carnoise`, so the cells here only install it, run it, and let you check the results. The output is the same as the earlier all-in-one notebook, and you can continue a folder that notebook started.

Before the first run, copy this repo folder to `MyDrive/carnoise-pipeline` (or change `PACKAGE_DIR` below).

More detail is in `README.md` and the `docs/` folder.

## 1. Mount Drive and install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PACKAGE_DIR = '/content/drive/MyDrive/carnoise-pipeline'
%pip -q install {PACKAGE_DIR}

## 2. Settings

The defaults come from `config/colab.toml`, and you can override them here. Settings that change the sound get locked to the output folder the first time it's used, so if you want to try something different, use a new `out_root` and a new `profile`.

In [ ]:
from dataclasses import replace
from pathlib import Path
from IPython.display import display
import carnoise
from carnoise import BatchRunner, load_config
from carnoise.batch import summarize

carnoise.configure_logging()
CONFIG = 'config/colab.toml'
cfg = load_config(f'{PACKAGE_DIR}/{CONFIG}')
cfg = replace(cfg,
    out_root=Path('/content/drive/MyDrive/ensense_denoise_carnoise_v3'),  # where results and logs go
    max_files=None,                                                     # None for everything, or e.g. 3 for a quick test
)
print('carnoise', carnoise.__version__)
print('Input: ', cfg.data_root)
print('Output:', cfg.out_root)
print('Profile:', cfg.profile, '| DSP:', cfg.dsp)

## 3. Pick the recordings

In [ ]:
runner = BatchRunner(cfg)
manifest = runner.prepare()
print('Manifest recordings:', len(manifest), '| units:', manifest.unit.nunique(),
      '| missing sources:', int((~manifest.source_found).sum()))
display(summarize(manifest))

## 4. Process everything

This goes through every recording one at a time, at about 5.5x real speed. If Colab disconnects, run all the cells again and it carries on where it stopped. A file that fails gets logged in `errors.csv`, and the run keeps going.

In [ ]:
stats = runner.run()
print(stats)

## 5. How far along is each unit?

In [ ]:
from carnoise.batch import completion_by_unit, failed_files
by_unit = completion_by_unit(manifest, cfg.out_root)
display(by_unit)
print(f'Complete: {int(by_unit.complete.sum())}/{int(by_unit.total.sum())}')
errors = failed_files(cfg.out_root)
if len(errors):
    print('Failed files:')
    display(errors.head(20))

## 6. Listen: before and after (both at the same volume)

In [ ]:
import numpy as np, soundfile as sf
from IPython.display import Audio
from carnoise.batch import safe_source
from carnoise.io import read_excerpt

LISTEN_INDEX = 0          # which recording (row number in the manifest)
LISTEN_SECONDS = 12
row = manifest.iloc[LISTEN_INDEX]
dst = Path(row.output_path)
if not dst.is_file():
    raise FileNotFoundError(f'Process this file first: {row.file}')
orig = read_excerpt(safe_source(cfg.data_root, row.file), LISTEN_SECONDS)
proc, sr = sf.read(str(dst), frames=int(LISTEN_SECONDS * sf.info(str(dst)).samplerate), always_2d=True, dtype='float32')
n = min(len(orig), len(proc))
gain = min(4.0, 0.95 / max(1e-8, float(np.max(np.abs(orig[:n])))))
print('File:', row.file, '| preview gain:', round(gain, 2))
print('Original:'); display(Audio((orig[:n] * gain).T, rate=sr, normalize=False))
print('Enhanced:'); display(Audio((proc[:n] * gain).T, rate=sr, normalize=False))

## Before you call it better

- Listen across several units, speeds and kinds of weather. Pay attention to voices, horns, passing cars, footsteps, birds and sirens. Quieter doesn't automatically mean better.
- Horns, sirens and alarms are kept: the hum-removal step only removes tones that stay steady for about 10 seconds. See `docs/algorithms.md`.
- `dsp_details.csv` shows what was decided for each file. A high-pass near 200 Hz means the file was mostly rumble.
- These recordings can contain private conversations. Keep the Drive folder restricted and don't share clips without permission.